Keshia-Lee Martin<br>
DX602 - Programming Toolkit for Data Science<br>
Final Project<br>
4/26/2026

<h2>Bike Rental Demand Analysis</h2>

**Introduction**

The purpose of this project is to complete an exploratory data analytics on a data set, answering questions I'd like to investigate. The dataset I've selected is Outdoor Recreation. This dataset analyzes bike rental patterns due to the season, month, and weekday. 

From this dataset, Here are some of the questions I'd like to answer:
1. How does weather, humidity, "feels like" temperature, actual temperature, and windspeed each affect the total amount of bike rentals? 
2. Which season has the highest number of bike rentals?
3. Which day of the week has the highest number of bike rentals?
4. Which weather category has the least amount of riders?
5. How do casual renters and registered renters differ?
6. How do the bike rentals change based on the traditional workdays versus the weekends?
7. What hour of the day has the most riders? 
8. What month and year has the most riders?
8. Are the bikes ridden more by the casual riders or the registered riders?

**Load and Inspecting the Data**

Pandas, numpy, matplotlib, and sklearn libraries will be used for the analysis.

In [241]:
# shared imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model
plt.close("all")

The data set is loaded in and reviewed by displaying 10 random samples. This helps to see how the data is organized, how it's formatted, and what columns and categories can be cleaned up or name changed.

In [242]:
# loading the dataset
bikes = pd.read_csv("bikes_final.csv")

# displaying 10 sample rows
bikes.sample(10)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week
4730,2011-11-10 04:00:00,4,0,1,1,15.58,19.695,94,6.0032,1,2,3,Thursday
2737,2011-07-02 23:00:00,3,0,0,1,29.52,33.335,54,19.0012,66,88,154,Saturday
2910,2011-07-10 04:00:00,3,0,0,1,27.06,31.060,69,6.0032,0,5,5,Sunday
4119,2011-10-03 16:00:00,4,0,1,2,16.40,20.455,76,7.0015,16,202,218,Monday
6929,2012-04-07 01:00:00,2,0,0,1,16.40,20.455,28,16.9979,9,60,69,Saturday
8595,2012-07-19 12:00:00,3,0,1,1,34.44,39.395,49,19.0012,59,215,274,Thursday
6120,2012-02-11 06:00:00,1,0,0,3,9.02,11.365,93,8.9981,1,8,9,Saturday
7845,2012-06-07 06:00:00,2,0,1,1,18.86,22.725,88,7.0015,5,165,170,Thursday
9021,2012-08-18 06:00:00,3,0,0,1,24.60,28.030,83,15.0013,4,23,27,Saturday
559,2011-02-06 11:00:00,1,0,0,1,13.12,15.150,49,16.9979,28,89,117,Sunday


Our column names are:
1. datetime - hourly date and the timestamp
2. season - seasons of the year, 1 = spring, 2 = summer, 3 = fall, 4 = winter 
3. holiday - whether the day is considered a holiday
4. workingday - whether the day is neither a weekend nor holiday
5. weather  - 
    1: Clear, Few clouds, Partly cloudy, Partly cloudy, 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist, 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds, 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog 
6. temp - temperature in Celsius
7. atemp - "feels like" temperature in Celsius
8. humidity - relative humidity
9. windspeed - wind speed
10. casual - number of non-registered user rentals
11. registered - number of registered user rentals
12. count - number of total rentals
13. day_of_week - days of the week

In [243]:
# checking columns
bikes.columns

Index(['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp',
       'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count',
       'day_of_week'],
      dtype='object')

The datatypes in our dataset are objects, integers, and floats.

In [244]:
# checking data types
bikes.dtypes

datetime        object
season           int64
holiday          int64
workingday       int64
weather          int64
temp           float64
atemp          float64
humidity         int64
windspeed      float64
casual           int64
registered       int64
count            int64
day_of_week     object
dtype: object

This dataset does not have any missing values.

In [245]:
# checking missing values
bikes.isna().any().any()


np.False_

In [246]:
bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   datetime     10886 non-null  object 
 1   season       10886 non-null  int64  
 2   holiday      10886 non-null  int64  
 3   workingday   10886 non-null  int64  
 4   weather      10886 non-null  int64  
 5   temp         10886 non-null  float64
 6   atemp        10886 non-null  float64
 7   humidity     10886 non-null  int64  
 8   windspeed    10886 non-null  float64
 9   casual       10886 non-null  int64  
 10  registered   10886 non-null  int64  
 11  count        10886 non-null  int64  
 12  day_of_week  10886 non-null  object 
dtypes: float64(3), int64(8), object(2)
memory usage: 1.1+ MB


**Data Cleaning and Preparation**

Creating a cleaned version of the data will improve the data analysis, making it easier to understand and read.

The datetime column is converted and separated into new columns by the hour, month, and year to do time analysis. The remapping will only be done for the month and year.


In [247]:
# converting the date column to pandas datetime
bikes["datetime"] = pd.to_datetime(bikes["datetime"])

# creating new columns that will be month, year, and hour
bikes["year"] = bikes["datetime"].dt.year.astype("category")
bikes["month"] = bikes["datetime"].dt.month.astype("category")
bikes["hour"] = bikes["datetime"].dt.hour.astype("category")

# remapping 'month' column to categorical
month_name = {
  1: "January",
  2: "February",
  3: "March",
  4: "April",
  5: "May",
  6: "June",
  7: "July",
  8: "August",
  9: "September",
  10: "October",
  11: "November",
  12: 'December'
}

bikes["month"] = bikes["month"].map(month_name)

#  remapping 'hour' column to categorical
hour_name = {
  0: "12am",
  1: "1am",
  2: "2am",
  3: "3am",
  4: "4am",
  5: "5am",
  6: "6am",
  7: "7am",
  8: "8am",
  9: "9am",
  10: "10am",
  11: "11am",
  12: "12pm",
  13:"1pm",
  14:"2pm",
  15:"3pm",
  16:"4pm",
  17:"5pm",
  18:"6pm",
  19:"7pm",
  20:"8pm",
  21:"9pm",
  22:"10pm",
  23:"11pm"
}

bikes["hour"] = bikes["hour"].map(hour_name)

# loading and view the table to make sure the columns are appearing as intended.
bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,4,0,1,1,15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm
10882,2012-12-19 20:00:00,4,0,1,1,14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm
10883,2012-12-19 21:00:00,4,0,1,1,13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm
10884,2012-12-19 22:00:00,4,0,1,1,13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm


The seasons are remapped from the numbers to their categorical names, with the type being changed to categorical.


In [248]:
# remapping of season 1, 2, 3, 4 to spring, summer, fall, and winter
season_name = {
  1: "Spring",
  2: "Summer",
  3: "Fall",
  4: "Winter"
}

# needs to be changed from an integer to a category
bikes["season"] = bikes["season"].map(season_name).astype("category")

# to check mapping change
bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour
0,2011-01-01 00:00:00,Spring,0,0,1,9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am
1,2011-01-01 01:00:00,Spring,0,0,1,9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am
2,2011-01-01 02:00:00,Spring,0,0,1,9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am
3,2011-01-01 03:00:00,Spring,0,0,1,9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am
4,2011-01-01 04:00:00,Spring,0,0,1,9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,Winter,0,1,1,15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm
10882,2012-12-19 20:00:00,Winter,0,1,1,14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm
10883,2012-12-19 21:00:00,Winter,0,1,1,13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm
10884,2012-12-19 22:00:00,Winter,0,1,1,13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm


The weather conditions are remapped from the numbers to their categorical names, with the type being changed to categorical.

In [249]:
# remapping of weather 1, 2, 3, 4 to weather condition categories
weather_name = {
  1: "Clear, Few clouds, Partly cloudy",
  2: "Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist",
  3: "Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds",
  4: "Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog"
}

# needs to be changed from an integer to a category
bikes["weather"] = bikes["weather"].map(weather_name).astype("category")

# to check mapping change
bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour
0,2011-01-01 00:00:00,Spring,0,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am
1,2011-01-01 01:00:00,Spring,0,0,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am
2,2011-01-01 02:00:00,Spring,0,0,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am
3,2011-01-01 03:00:00,Spring,0,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am
4,2011-01-01 04:00:00,Spring,0,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,Winter,0,1,"Clear, Few clouds, Partly cloudy",15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm
10882,2012-12-19 20:00:00,Winter,0,1,"Clear, Few clouds, Partly cloudy",14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm
10883,2012-12-19 21:00:00,Winter,0,1,"Clear, Few clouds, Partly cloudy",13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm
10884,2012-12-19 22:00:00,Winter,0,1,"Clear, Few clouds, Partly cloudy",13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm


Changing the day_of_week column to categorical, but also changing the order to be Monday-Sunday, not Saturday-Friday.

In [250]:
# changing day of week type to category and reordering them from S-F to M-S
week_order = ["Monday",
  "Tuesday",
  "Wednesday",
  "Thursday",
  "Friday",
  "Saturday",
  "Sunday"]
  
bikes["day_of_week"] = pd.Categorical(bikes["day_of_week"], categories=week_order, ordered=True)

# verification that the new order and categories are correct
bikes["day_of_week"].unique()


['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
Categories (7, object): ['Monday' < 'Tuesday' < 'Wednesday' < 'Thursday' < 'Friday' < 'Saturday' < 'Sunday']

The holiday and workingday is changed from numerical to categorical strings and types.

In [251]:
#changing numerical to categorical for the holiday column
holiday_name = {
  0: "Not a Holiday",
  1: "Holiday"
}

bikes["holiday"] = bikes["holiday"].map(holiday_name).astype("category")

bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour
0,2011-01-01 00:00:00,Spring,Not a Holiday,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am
1,2011-01-01 01:00:00,Spring,Not a Holiday,0,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am
2,2011-01-01 02:00:00,Spring,Not a Holiday,0,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am
3,2011-01-01 03:00:00,Spring,Not a Holiday,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am
4,2011-01-01 04:00:00,Spring,Not a Holiday,0,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,Winter,Not a Holiday,1,"Clear, Few clouds, Partly cloudy",15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm
10882,2012-12-19 20:00:00,Winter,Not a Holiday,1,"Clear, Few clouds, Partly cloudy",14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm
10883,2012-12-19 21:00:00,Winter,Not a Holiday,1,"Clear, Few clouds, Partly cloudy",13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm
10884,2012-12-19 22:00:00,Winter,Not a Holiday,1,"Clear, Few clouds, Partly cloudy",13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm


In [252]:
# changing numerical to categorical for the working day column
workingday_name = {
  0: "Not a Workday",
  1: "Workday"
}

bikes["workingday"] = bikes["workingday"].map(workingday_name).astype("category")

# verification that all columns have been updated as intended
bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour
0,2011-01-01 00:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am
1,2011-01-01 01:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am
2,2011-01-01 02:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am
3,2011-01-01 03:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am
4,2011-01-01 04:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm
10882,2012-12-19 20:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm
10883,2012-12-19 21:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm
10884,2012-12-19 22:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm


Creating a new column that determines if a day is a weekend or a weekday.

In [253]:
# creating a new column that takes the Saturday and Sundays of the day of the week and labels it as Weekend.
# if it is not Saturday or Sunday, its labeled Weekday
bikes["weekend_weekday"] = np.where(bikes["day_of_week"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
bikes["weekend_weekday"] = bikes["weekend_weekday"].astype("category")

bikes

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,day_of_week,year,month,hour,weekend_weekday
0,2011-01-01 00:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,81,0.0000,3,13,16,Saturday,2011,January,12am,Weekend
1,2011-01-01 01:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,8,32,40,Saturday,2011,January,1am,Weekend
2,2011-01-01 02:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.02,13.635,80,0.0000,5,27,32,Saturday,2011,January,2am,Weekend
3,2011-01-01 03:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,3,10,13,Saturday,2011,January,3am,Weekend
4,2011-01-01 04:00:00,Spring,Not a Holiday,Not a Workday,"Clear, Few clouds, Partly cloudy",9.84,14.395,75,0.0000,0,1,1,Saturday,2011,January,4am,Weekend
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,2012-12-19 19:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",15.58,19.695,50,26.0027,7,329,336,Wednesday,2012,December,7pm,Weekday
10882,2012-12-19 20:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",14.76,17.425,57,15.0013,10,231,241,Wednesday,2012,December,8pm,Weekday
10883,2012-12-19 21:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",13.94,15.910,61,15.0013,4,164,168,Wednesday,2012,December,9pm,Weekday
10884,2012-12-19 22:00:00,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",13.94,17.425,61,6.0032,12,117,129,Wednesday,2012,December,10pm,Weekday


**Descriptive Statistics**

Summarizing the statistics for numerical columns and the categorical columns.

In [254]:
# descriptive statistics for numerical columns
summary_numerical = bikes.describe(include="number")
summary_numerical

,temp,atemp,humidity,windspeed,casual,registered,count
count,10886.00000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000
mean,20.23086,23.655084,61.886460,12.799395,36.021955,155.552177,191.574132
std,7.79159,8.474601,19.245033,8.164537,49.960477,151.039033,181.144454
min,0.82000,0.760000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,13.94000,16.665000,47.000000,7.001500,4.000000,36.000000,42.000000
50%,20.50000,24.240000,62.000000,12.998000,17.000000,118.000000,145.000000
75%,26.24000,31.060000,77.000000,16.997900,49.000000,222.000000,284.000000
max,41.00000,45.455000,100.000000,56.996900,367.000000,886.000000,977.000000


There are 7 numerical columns. They are temp, atemp, humidity, windspeed, casual, registered, and count. There are 10,886 lines of data and as noted before, no data values are missing.

In [255]:
# summary for categorical columns
summary_categorical = bikes.describe(include="category")
summary_categorical

,season,holiday,workingday,weather,day_of_week,year,month,hour,weekend_weekday
count,10886,10886,10886,10886,10886,10886,10886,10886,10886
unique,4,2,2,4,7,2,12,24,2
top,Winter,Not a Holiday,Workday,"Clear, Few clouds, Partly cloudy",Saturday,2012,May,12pm,Weekday
freq,2734,10575,7412,7192,1584,5464,912,456,7723


There are 9 categorical columns. They are season, holiday, workingday, weather, day_of_week, year, month, hour, and weekend_weekday. There are 10,886 lines of data and as noted before, no data values are missing.

Using this table, it's confirmed that the correct number of seasons, days of the week, months, and hours are in the data. It's also verifies that the data is appearing and has been cleaned up as intended by looking at holiday and weekend_weekday. For weekend_weekday, there are more weekdays than weekends. For holiday, there are more days without holidays than there are with.

**3. Numeric Data Visualizations**

Create charts that illustrate the distribution of numerical data, such as:

* Histograms
* Box plots
* Bar charts (when appropriate)
* Create charts showing distributions
* Explain what the charts reveal

In [256]:
# histograms - for loop to do all at once


# box plots - for loop to do all at once





**Section 6: Categorical Data Analysis**

* Create charts for categorical variables
* Discuss the results
* Show the distribution of one or more categorical variables
* Use charts such as:
    * Bar charts
    * Pie charts

**Section 7: Grouped Analysis**

* Group by a meaningful category
* Compute grouped summary statistics
* Visualize grouped comparisons

* Group data by one or more categorical variables
* Compute descriptive statistics for grouped data
* Create charts to visualize grouped comparisons

**Section 8: New Feature Engineering**

* Create at least one new column
* Explain why it is useful
* Use it in a chart or summary

Create at least **one meaningful new column** for your analysis.

Examples:

* A ratio between two existing columns
* A percentage
* A standardized value relative to a base year or base category

You must use this new column somewhere in your analysis.

**Section 9: Trends Over Time**

* Plot one or more time-based trends
* Include standardized values if useful for comparison
* Use line charts to illustrate trends over time
* Use standardized values when appropriate so multiple variables can be compared on the same scale

**Section 10: Variable Relationships**

* Compute correlation or covariance
* Make scatter plots
* Discuss possible relationships

* Investigate possible relationships between features
* Use:
    * Correlation or covariance statistics
    * Scatter plots

**Section 11: Hypothesis**

* State your hypothesis about a relationship in the data

**Section 12: Linear Regression**

* Select features and outcome variable
* Train the regression model
* Display predictions vs. actual values
* Plot the regression line
* Interpret results

Use `sklearn` to build a **linear regression model** that attempts to explain a relationship between one outcome variable and one or more features.

Your regression section must include:

* A clear choice of predictor variable(s)
* A clear outcome variable
* Model fitting using `LinearRegression`
* A plot showing predicted values compared to actual values
* A regression line

**11. Regression Discussion**

Briefly explain:

* What the regression results suggest
* Whether the model does a good job explaining the relationship
* Any limitations of the model


**Section 13: Reflection / Conclusion**

* Summarize your findings
* Explain what you learned
* Mention limitations and possible future improvements

<h2>**13. Something New**</h2>

You must learn and apply **at least one skill that goes beyond class examples and homework**.

Examples might include:

* A new pandas technique
* A more advanced matplotlib customization
* A new type of grouping or transformation
* A more polished plot layout

**12. Code Quality and Reuse**

Identify repetitive parts of your code and simplify them using:

* Helper functions
* Loops
* Reusable code blocks